<a href="https://colab.research.google.com/github/zhangling297/MAT_CS_599_deepLearningClassPracitices/blob/main/copy_(2)_of_Mat599_FinalProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Downloads the NHANES files
## Builds the analytic dataset
### Produces the three comparison tables.

#### Setup
#1. The CDC P_DEMO file provides demographics plus the special 2017–March 2020 weights WTINTPRP and WTMECPRP
## The design variables SDMVSTRA and SDMVPSU; P_BMX contains BMXBMI; and P_DIQ contains DIQ010 for diagnosed diabetes.
### For 2017–March 2020 pre-pandemic NHANES analyses, CDC says to use the special pre-pandemic weights;
####This project merges questionnaire and examination data, WTMECPRP is the appropriate weight to use.

##### Project steps

# - Download P_DEMO.xpt, P_BMX.xpt, and P_DIQ.xpt.
#  - Read each XPT file into pandas.
# - Merge them by SEQN.
# - Keep adults aged 18+.

# - Create outcome indicators:

#  - obesity = 1 if BMXBMI >= 30

# - diabetes = 1 if DIQ010 == 1

# Create demographic groups:

# age groups, sex , race/ethnicity

# Use WTMECPRP to compute weighted prevalence.

# Build: Table 1 Demographic distribution comparison

# Table 2 Risk factors in clinical and person comparison

# Table 3 Summary comparison

In [ ]:
# ============================================================
# NHANES 2017-March 2020 class project:
# Comparative tables for diabetes vs obesity
#
# Files used:
#   P_DEMO.xpt : demographics + survey design + weights
#   P_BMX.xpt  : body measures (includes BMI)
#   P_DIQ.xpt  : diabetes questionnaire
#
# Outputs:
#   - table1_demographic_distribution.csv
#   - table2_risk_factors.csv
#   - table3_summary_comparison.csv
# ============================================================

import os
import requests
from io import BytesIO
import pandas as pd
import numpy as np

# -----------------------------
# 1. CDC file URLs
# -----------------------------
URLS = {
    "P_DEMO": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_DEMO.xpt",
    "P_BMX":  "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_BMX.xpt",
    "P_DIQ":  "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_DIQ.xpt",
}

DATA_DIR = "nhanes_data"
os.makedirs(DATA_DIR, exist_ok=True)

# -----------------------------
# 2. Downloader
# -----------------------------
def download_file(url: str, out_path: str) -> None:
    """Download file from CDC if not already present."""
    if os.path.exists(out_path):
        print(f"Already exists: {out_path}")
        return

    print(f"Downloading {url} ...")
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    with open(out_path, "wb") as f:
        f.write(r.content)
    print(f"Saved to {out_path}")

# Download all files
for name, url in URLS.items():
    download_file(url, os.path.join(DATA_DIR, f"{name}.xpt"))

# -----------------------------
# 3. Reader
# -----------------------------
def read_xpt(path: str) -> pd.DataFrame:
    """Read SAS transport (.xpt) file into pandas."""
    return pd.read_sas(path, format="xport")

demo = read_xpt(os.path.join(DATA_DIR, "P_DEMO.xpt"))
bmx  = read_xpt(os.path.join(DATA_DIR, "P_BMX.xpt"))
diq  = read_xpt(os.path.join(DATA_DIR, "P_DIQ.xpt"))

# -----------------------------
# 4. Keep needed columns only
# -----------------------------
demo_cols = [
    "SEQN",       # respondent ID
    "RIDAGEYR",   # age in years
    "RIAGENDR",   # sex
    "RIDRETH3",   # race/ethnicity
    "RIDSTATR",   # interview/exam status
    "WTMECPRP",   # MEC exam weight for 2017-Mar 2020
    "WTINTPRP",   # interview weight
    "SDMVSTRA",   # stratum
    "SDMVPSU"     # PSU
]
bmx_cols = [
    "SEQN",
    "BMXBMI",     # BMI
    "BMXWAIST"    # waist circumference
]
diq_cols = [
    "SEQN",
    "DIQ010"      # ever told by doctor has diabetes
]

demo = demo[demo_cols].copy()
bmx  = bmx[bmx_cols].copy()
diq  = diq[diq_cols].copy()

# -----------------------------
# 5. Merge datasets
# -----------------------------
df = demo.merge(bmx, on="SEQN", how="left").merge(diq, on="SEQN", how="left")

# -----------------------------
# 6. Restrict to adults
# -----------------------------
df = df[df["RIDAGEYR"] >= 18].copy()

# Optional:
# if you want to use only participants with MEC exam status:
# RIDSTATR == 2 means interviewed + examined
df = df[df["RIDSTATR"] == 2].copy()

# -----------------------------
# 7. Clean values / create outcomes
# -----------------------------
# Convert to numeric just in case
for col in ["RIDAGEYR", "RIAGENDR", "RIDRETH3", "WTMECPRP", "BMXBMI", "BMXWAIST", "DIQ010"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Outcome definitions
df["obesity"] = np.where(df["BMXBMI"] >= 30, 1, np.where(df["BMXBMI"].notna(), 0, np.nan))
df["diabetes"] = np.where(df["DIQ010"] == 1, 1, np.where(df["DIQ010"].isin([2, 3]), 0, np.nan))
# Note:
# NHANES DIQ010 coding commonly uses:
#   1 = Yes
#   2 = No
#   3 = Borderline
# For a simple class project, this script treats borderline as not diagnosed diabetes.
# You can instead exclude DIQ010 == 3 if your instructor prefers.

# Clinical / person-level risk indicators for comparison table
df["high_waist"] = np.where(
    ((df["RIAGENDR"] == 1) & (df["BMXWAIST"] > 102)) |
    ((df["RIAGENDR"] == 2) & (df["BMXWAIST"] > 88)),
    1,
    np.where(df["BMXWAIST"].notna() & df["RIAGENDR"].notna(), 0, np.nan)
)

# -----------------------------
# 8. Recode groups
# -----------------------------
def age_group(age):
    if pd.isna(age):
        return np.nan
    if 18 <= age <= 39:
        return "18-39"
    elif 40 <= age <= 59:
        return "40-59"
    else:
        return "60+"

sex_map = {
    1: "Male",
    2: "Female"
}

race_map = {
    1: "Mexican American",
    2: "Other Hispanic",
    3: "Non-Hispanic White",
    4: "Non-Hispanic Black",
    6: "Non-Hispanic Asian",
    7: "Other / Multiracial"
}

df["age_group"] = df["RIDAGEYR"].apply(age_group)
df["sex"] = df["RIAGENDR"].map(sex_map)
df["race_eth"] = df["RIDRETH3"].map(race_map)

# -----------------------------
# 9. Weighted helpers
# -----------------------------
WEIGHT = "WTMECPRP"

def weighted_prevalence(data: pd.DataFrame, outcome: str, weight: str = WEIGHT) -> float:
    """Weighted prevalence (%) among non-missing observations."""
    temp = data[[outcome, weight]].dropna()
    if temp.empty:
        return np.nan
    return 100 * (temp[outcome] * temp[weight]).sum() / temp[weight].sum()

def weighted_mean(data: pd.DataFrame, var: str, weight: str = WEIGHT) -> float:
    """Weighted mean for continuous variables."""
    temp = data[[var, weight]].dropna()
    if temp.empty:
        return np.nan
    return np.average(temp[var], weights=temp[weight])

def weighted_numerator_estimate(data: pd.DataFrame, outcome: str, weight: str = WEIGHT) -> float:
    """Weighted estimate of number with outcome (population estimate)."""
    temp = data[[outcome, weight]].dropna()
    if temp.empty:
        return np.nan
    return (temp[outcome] * temp[weight]).sum()

def weighted_denominator_estimate(data: pd.DataFrame, outcome: str, weight: str = WEIGHT) -> float:
    """Weighted population denominator used in the prevalence estimate."""
    temp = data[[outcome, weight]].dropna()
    if temp.empty:
        return np.nan
    return temp[weight].sum()

# -----------------------------
# 10. Table 1
# Demographic distribution comparison
# -----------------------------
table1_rows = []

# Total
table1_rows.append({
    "Group": "Total adults",
    "Unweighted_n_diabetes": df["diabetes"].notna().sum(),
    "Unweighted_n_obesity": df["obesity"].notna().sum(),
    "Weighted_pop_for_diabetes": weighted_denominator_estimate(df, "diabetes"),
    "Weighted_pop_for_obesity": weighted_denominator_estimate(df, "obesity"),
    "Diabetes_prevalence_pct": weighted_prevalence(df, "diabetes"),
    "Obesity_prevalence_pct": weighted_prevalence(df, "obesity"),
})

# By age group
for g in ["18-39", "40-59", "60+"]:
    sub = df[df["age_group"] == g]
    table1_rows.append({
        "Group": f"Age {g}",
        "Unweighted_n_diabetes": sub["diabetes"].notna().sum(),
        "Unweighted_n_obesity": sub["obesity"].notna().sum(),
        "Weighted_pop_for_diabetes": weighted_denominator_estimate(sub, "diabetes"),
        "Weighted_pop_for_obesity": weighted_denominator_estimate(sub, "obesity"),
        "Diabetes_prevalence_pct": weighted_prevalence(sub, "diabetes"),
        "Obesity_prevalence_pct": weighted_prevalence(sub, "obesity"),
    })

# By sex
for g in ["Male", "Female"]:
    sub = df[df["sex"] == g]
    table1_rows.append({
        "Group": g,
        "Unweighted_n_diabetes": sub["diabetes"].notna().sum(),
        "Unweighted_n_obesity": sub["obesity"].notna().sum(),
        "Weighted_pop_for_diabetes": weighted_denominator_estimate(sub, "diabetes"),
        "Weighted_pop_for_obesity": weighted_denominator_estimate(sub, "obesity"),
        "Diabetes_prevalence_pct": weighted_prevalence(sub, "diabetes"),
        "Obesity_prevalence_pct": weighted_prevalence(sub, "obesity"),
    })

# By race/ethnicity
race_order = [
    "Mexican American",
    "Other Hispanic",
    "Non-Hispanic White",
    "Non-Hispanic Black",
    "Non-Hispanic Asian",
    "Other / Multiracial"
]

for g in race_order:
    sub = df[df["race_eth"] == g]
    table1_rows.append({
        "Group": g,
        "Unweighted_n_diabetes": sub["diabetes"].notna().sum(),
        "Unweighted_n_obesity": sub["obesity"].notna().sum(),
        "Weighted_pop_for_diabetes": weighted_denominator_estimate(sub, "diabetes"),
        "Weighted_pop_for_obesity": weighted_denominator_estimate(sub, "obesity"),
        "Diabetes_prevalence_pct": weighted_prevalence(sub, "diabetes"),
        "Obesity_prevalence_pct": weighted_prevalence(sub, "obesity"),
    })

table1 = pd.DataFrame(table1_rows)

# -----------------------------
# 11. Table 2
# Risk factors in clinical and person comparison
# -----------------------------
table2_rows = []

# Diabetes group vs non-diabetes
for label, condition in [
    ("Has diabetes", df["diabetes"] == 1),
    ("No diabetes", df["diabetes"] == 0),
    ("Has obesity", df["obesity"] == 1),
    ("No obesity", df["obesity"] == 0),
]:
    sub = df[condition].copy()

    table2_rows.append({
        "Comparison_group": label,
        "Unweighted_n": len(sub),
        "Weighted_population": sub[WEIGHT].sum(skipna=True),
        "Mean_age_years": weighted_mean(sub, "RIDAGEYR"),
        "Mean_BMI": weighted_mean(sub, "BMXBMI"),
        "High_waist_pct": weighted_prevalence(sub, "high_waist"),
        "Female_pct": weighted_prevalence(
            sub.assign(is_female=np.where(sub["sex"] == "Female", 1, 0)),
            "is_female"
        ),
        "Age_60plus_pct": weighted_prevalence(
            sub.assign(age60=np.where(sub["RIDAGEYR"] >= 60, 1, 0)),
            "age60"
        ),
        "Obesity_pct_within_group": weighted_prevalence(sub, "obesity"),
        "Diabetes_pct_within_group": weighted_prevalence(sub, "diabetes"),
    })

table2 = pd.DataFrame(table2_rows)

# -----------------------------
# 12. Table 3
# Summary comparison
# -----------------------------
# For a simple class project, keep this table NHANES-based
# so one source is used consistently.
table3 = pd.DataFrame([
    {
        "Indicator": "Adult analytic sample (unweighted)",
        "Diabetes": int(df["diabetes"].notna().sum()),
        "Obesity": int(df["obesity"].notna().sum()),
    },
    {
        "Indicator": "Weighted adult population denominator",
        "Diabetes": weighted_denominator_estimate(df, "diabetes"),
        "Obesity": weighted_denominator_estimate(df, "obesity"),
    },
    {
        "Indicator": "Weighted prevalence (%)",
        "Diabetes": weighted_prevalence(df, "diabetes"),
        "Obesity": weighted_prevalence(df, "obesity"),
    },
    {
        "Indicator": "Weighted population estimate with condition",
        "Diabetes": weighted_numerator_estimate(df, "diabetes"),
        "Obesity": weighted_numerator_estimate(df, "obesity"),
    },
    {
        "Indicator": "Mean age (years)",
        "Diabetes": weighted_mean(df[df["diabetes"] == 1], "RIDAGEYR"),
        "Obesity": weighted_mean(df[df["obesity"] == 1], "RIDAGEYR"),
    },
    {
        "Indicator": "Mean BMI",
        "Diabetes": weighted_mean(df[df["diabetes"] == 1], "BMXBMI"),
        "Obesity": weighted_mean(df[df["obesity"] == 1], "BMXBMI"),
    },
    {
        "Indicator": "High waist circumference prevalence (%)",
        "Diabetes": weighted_prevalence(df[df["diabetes"] == 1], "high_waist"),
        "Obesity": weighted_prevalence(df[df["obesity"] == 1], "high_waist"),
    }
])

# -----------------------------
# 13. Round and save outputs
# -----------------------------
def round_numeric(df_in: pd.DataFrame, digits: int = 1) -> pd.DataFrame:
    df_out = df_in.copy()
    num_cols = df_out.select_dtypes(include=[np.number]).columns
    df_out[num_cols] = df_out[num_cols].round(digits)
    return df_out

table1_out = round_numeric(table1, 1)
table2_out = round_numeric(table2, 1)
table3_out = round_numeric(table3, 1)

table1_out.to_csv("table1_demographic_distribution.csv", index=False)
table2_out.to_csv("table2_risk_factors.csv", index=False)
table3_out.to_csv("table3_summary_comparison.csv", index=False)

print("\n=== Table 1. Demographic distribution comparison ===")
print(table1_out.to_string(index=False))

print("\n=== Table 2. Risk factors in clinical and person comparison ===")
print(table2_out.to_string(index=False))

print("\n=== Table 3. Summary comparison ===")
print(table3_out.to_string(index=False))

print("\nSaved:")
print(" - table1_demographic_distribution.csv")
print(" - table2_risk_factors.csv")
print(" - table3_summary_comparison.csv")

In [ ]:
# ============================================================
# GOOGLE COLAB FULL SCRIPT
# NHANES Diabetes and Obesity Comparison Project
#
# Tasks:
# 1. Download NHANES data and save 3 comparison tables
# 2. Create a copy of this script + README.md
# 3. Push files to your GitHub repository
# ============================================================

# =========================
# 0. Install packages
# =========================
!pip -q install pandas requests openpyxl

import os
import requests
from io import BytesIO
import pandas as pd
import numpy as np
import textwrap
import subprocess
from google.colab import files

# =========================
# 1. Project settings
# =========================
PROJECT_NAME = "nhanes-diabetes-obesity-project"
OUTPUT_DIR = os.path.join(PROJECT_NAME, "outputs")
DATA_DIR = os.path.join(PROJECT_NAME, "data")
SCRIPT_NAME = "nhanes_diabetes_obesity_project.py"
README_NAME = "README.md"

os.makedirs(PROJECT_NAME, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

# =========================
# 2. NHANES file URLs
# =========================
URLS = {
    "P_DEMO": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_DEMO.xpt",
    "P_BMX":  "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_BMX.xpt",
    "P_DIQ":  "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_DIQ.xpt",
}

# =========================
# 3. Download NHANES files
# =========================
def download_file(url: str, out_path: str) -> None:
    if os.path.exists(out_path):
        print(f"Already exists: {out_path}")
        return
    print(f"Downloading: {url}")
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    with open(out_path, "wb") as f:
        f.write(r.content)
    print(f"Saved: {out_path}")

for name, url in URLS.items():
    download_file(url, os.path.join(DATA_DIR, f"{name}.xpt"))

# =========================
# 4. Read XPT files
# =========================
def read_xpt(path: str) -> pd.DataFrame:
    return pd.read_sas(path, format="xport")

demo = read_xpt(os.path.join(DATA_DIR, "P_DEMO.xpt"))
bmx  = read_xpt(os.path.join(DATA_DIR, "P_BMX.xpt"))
diq  = read_xpt(os.path.join(DATA_DIR, "P_DIQ.xpt"))

print("Loaded files:")
print("DEMO:", demo.shape)
print("BMX :", bmx.shape)
print("DIQ :", diq.shape)

# =========================
# 5. Keep needed columns
# =========================
demo_cols = [
    "SEQN",
    "RIDAGEYR",
    "RIAGENDR",
    "RIDRETH3",
    "RIDSTATR",
    "WTMECPRP",
    "WTINTPRP",
    "SDMVSTRA",
    "SDMVPSU"
]

bmx_cols = [
    "SEQN",
    "BMXBMI",
    "BMXWAIST"
]

diq_cols = [
    "SEQN",
    "DIQ010"
]

demo = demo[demo_cols].copy()
bmx  = bmx[bmx_cols].copy()
diq  = diq[diq_cols].copy()

# =========================
# 6. Merge datasets
# =========================
df = demo.merge(bmx, on="SEQN", how="left").merge(diq, on="SEQN", how="left")
print("Merged shape:", df.shape)

# =========================
# 7. Restrict sample
# =========================
df = df[df["RIDAGEYR"] >= 18].copy()
df = df[df["RIDSTATR"] == 2].copy()   # interviewed + MEC examined
print("Adult MEC sample shape:", df.shape)

# =========================
# 8. Clean and define variables
# =========================
for col in ["RIDAGEYR", "RIAGENDR", "RIDRETH3", "WTMECPRP", "BMXBMI", "BMXWAIST", "DIQ010"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Outcomes
df["obesity"] = np.where(df["BMXBMI"] >= 30, 1,
                         np.where(df["BMXBMI"].notna(), 0, np.nan))

# Simple class-project rule:
# 1 = yes diabetes
# 2 = no
# 3 = borderline -> treated as 0 here
df["diabetes"] = np.where(df["DIQ010"] == 1, 1,
                          np.where(df["DIQ010"].isin([2, 3]), 0, np.nan))

# High waist circumference
df["high_waist"] = np.where(
    ((df["RIAGENDR"] == 1) & (df["BMXWAIST"] > 102)) |
    ((df["RIAGENDR"] == 2) & (df["BMXWAIST"] > 88)),
    1,
    np.where(df["BMXWAIST"].notna() & df["RIAGENDR"].notna(), 0, np.nan)
)

# Age groups
def age_group(age):
    if pd.isna(age):
        return np.nan
    if 18 <= age <= 39:
        return "18-39"
    elif 40 <= age <= 59:
        return "40-59"
    else:
        return "60+"

df["age_group"] = df["RIDAGEYR"].apply(age_group)

# Sex labels
sex_map = {1: "Male", 2: "Female"}
df["sex"] = df["RIAGENDR"].map(sex_map)

# Race/ethnicity labels
race_map = {
    1: "Mexican American",
    2: "Other Hispanic",
    3: "Non-Hispanic White",
    4: "Non-Hispanic Black",
    6: "Non-Hispanic Asian",
    7: "Other / Multiracial"
}
df["race_eth"] = df["RIDRETH3"].map(race_map)

# =========================
# 9. Weighted helper functions
# =========================
WEIGHT = "WTMECPRP"

def weighted_prevalence(data: pd.DataFrame, outcome: str, weight: str = WEIGHT) -> float:
    temp = data[[outcome, weight]].dropna()
    if temp.empty:
        return np.nan
    return 100 * (temp[outcome] * temp[weight]).sum() / temp[weight].sum()

def weighted_mean(data: pd.DataFrame, var: str, weight: str = WEIGHT) -> float:
    temp = data[[var, weight]].dropna()
    if temp.empty:
        return np.nan
    return np.average(temp[var], weights=temp[weight])

def weighted_denominator(data: pd.DataFrame, outcome: str, weight: str = WEIGHT) -> float:
    temp = data[[outcome, weight]].dropna()
    if temp.empty:
        return np.nan
    return temp[weight].sum()

def weighted_numerator(data: pd.DataFrame, outcome: str, weight: str = WEIGHT) -> float:
    temp = data[[outcome, weight]].dropna()
    if temp.empty:
        return np.nan
    return (temp[outcome] * temp[weight]).sum()

# =========================
# 10. Table 1
# Demographic distribution comparison
# =========================
table1_rows = []

def add_table1_row(label, sub):
    table1_rows.append({
        "Group": label,
        "Unweighted_n_diabetes": sub["diabetes"].notna().sum(),
        "Unweighted_n_obesity": sub["obesity"].notna().sum(),
        "Weighted_pop_diabetes": weighted_denominator(sub, "diabetes"),
        "Weighted_pop_obesity": weighted_denominator(sub, "obesity"),
        "Diabetes_prevalence_pct": weighted_prevalence(sub, "diabetes"),
        "Obesity_prevalence_pct": weighted_prevalence(sub, "obesity"),
    })

add_table1_row("Total adults", df)

for g in ["18-39", "40-59", "60+"]:
    add_table1_row(f"Age {g}", df[df["age_group"] == g])

for g in ["Male", "Female"]:
    add_table1_row(g, df[df["sex"] == g])

for g in [
    "Mexican American",
    "Other Hispanic",
    "Non-Hispanic White",
    "Non-Hispanic Black",
    "Non-Hispanic Asian",
    "Other / Multiracial"
]:
    add_table1_row(g, df[df["race_eth"] == g])

table1 = pd.DataFrame(table1_rows)

# =========================
# 11. Table 2
# Risk factors in clinical and person comparison
# =========================
table2_rows = []

comparison_groups = [
    ("Has diabetes", df["diabetes"] == 1),
    ("No diabetes", df["diabetes"] == 0),
    ("Has obesity", df["obesity"] == 1),
    ("No obesity", df["obesity"] == 0),
]

for label, mask in comparison_groups:
    sub = df[mask].copy()
    sub["is_female"] = np.where(sub["sex"] == "Female", 1, 0)
    sub["age60plus"] = np.where(sub["RIDAGEYR"] >= 60, 1, 0)

    table2_rows.append({
        "Comparison_group": label,
        "Unweighted_n": len(sub),
        "Weighted_population": sub[WEIGHT].sum(skipna=True),
        "Mean_age_years": weighted_mean(sub, "RIDAGEYR"),
        "Mean_BMI": weighted_mean(sub, "BMXBMI"),
        "High_waist_pct": weighted_prevalence(sub, "high_waist"),
        "Female_pct": weighted_prevalence(sub, "is_female"),
        "Age_60plus_pct": weighted_prevalence(sub, "age60plus"),
        "Obesity_pct_within_group": weighted_prevalence(sub, "obesity"),
        "Diabetes_pct_within_group": weighted_prevalence(sub, "diabetes"),
    })

table2 = pd.DataFrame(table2_rows)

# =========================
# 12. Table 3
# Summary comparison
# =========================
table3 = pd.DataFrame([
    {
        "Indicator": "Adult analytic sample (unweighted)",
        "Diabetes": df["diabetes"].notna().sum(),
        "Obesity": df["obesity"].notna().sum(),
    },
    {
        "Indicator": "Weighted adult population denominator",
        "Diabetes": weighted_denominator(df, "diabetes"),
        "Obesity": weighted_denominator(df, "obesity"),
    },
    {
        "Indicator": "Weighted prevalence (%)",
        "Diabetes": weighted_prevalence(df, "diabetes"),
        "Obesity": weighted_prevalence(df, "obesity"),
    },
    {
        "Indicator": "Weighted population estimate with condition",
        "Diabetes": weighted_numerator(df, "diabetes"),
        "Obesity": weighted_numerator(df, "obesity"),
    },
    {
        "Indicator": "Mean age (years) among cases",
        "Diabetes": weighted_mean(df[df["diabetes"] == 1], "RIDAGEYR"),
        "Obesity": weighted_mean(df[df["obesity"] == 1], "RIDAGEYR"),
    },
    {
        "Indicator": "Mean BMI among cases",
        "Diabetes": weighted_mean(df[df["diabetes"] == 1], "BMXBMI"),
        "Obesity": weighted_mean(df[df["obesity"] == 1], "BMXBMI"),
    },
    {
        "Indicator": "High waist circumference prevalence (%) among cases",
        "Diabetes": weighted_prevalence(df[df["diabetes"] == 1], "high_waist"),
        "Obesity": weighted_prevalence(df[df["obesity"] == 1], "high_waist"),
    },
])

# =========================
# 13. Round outputs
# =========================
def round_numeric(df_in, digits=1):
    out = df_in.copy()
    num_cols = out.select_dtypes(include=[np.number]).columns
    out[num_cols] = out[num_cols].round(digits)
    return out

table1_out = round_numeric(table1, 1)
table2_out = round_numeric(table2, 1)
table3_out = round_numeric(table3, 1)

# =========================
# 14. Save tables locally in Colab
# Task 1
# =========================
table1_csv = os.path.join(OUTPUT_DIR, "table1_demographic_distribution.csv")
table2_csv = os.path.join(OUTPUT_DIR, "table2_risk_factors.csv")
table3_csv = os.path.join(OUTPUT_DIR, "table3_summary_comparison.csv")
analytic_csv = os.path.join(OUTPUT_DIR, "analytic_dataset.csv")
excel_path = os.path.join(OUTPUT_DIR, "nhanes_diabetes_obesity_tables.xlsx")

table1_out.to_csv(table1_csv, index=False)
table2_out.to_csv(table2_csv, index=False)
table3_out.to_csv(table3_csv, index=False)
df.to_csv(analytic_csv, index=False)

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    table1_out.to_excel(writer, sheet_name="Table1_Demographic", index=False)
    table2_out.to_excel(writer, sheet_name="Table2_RiskFactors", index=False)
    table3_out.to_excel(writer, sheet_name="Table3_Summary", index=False)

print("\nSaved local files:")
print(table1_csv)
print(table2_csv)
print(table3_csv)
print(analytic_csv)
print(excel_path)

# =========================
# 15. Create README.md
# =========================
readme_text = f"""# NHANES Diabetes and Obesity Comparison Project

## Project Overview
This Google Colab project compares diabetes and obesity in U.S. adults using NHANES 2017-March 2020 pre-pandemic data.

## Research Tasks
1. Save the comparison tables used in the class project
2. Create a reusable script and README for GitHub

## Data Files Used
- `P_DEMO.xpt` for demographics and survey weights
- `P_BMX.xpt` for BMI and waist circumference
- `P_DIQ.xpt` for diagnosed diabetes questionnaire data

## Variables Used
### Demographic variables
- `RIDAGEYR`: age in years
- `RIAGENDR`: sex
- `RIDRETH3`: race/ethnicity

### Survey design variables
- `WTMECPRP`: MEC exam weight
- `WTINTPRP`: interview weight
- `SDMVSTRA`: stratum
- `SDMVPSU`: PSU

### Clinical variables
- `BMXBMI`: body mass index
- `BMXWAIST`: waist circumference
- `DIQ010`: doctor told you have diabetes

## Analytic Definitions
- Adults only: `RIDAGEYR >= 18`
- MEC examined sample: `RIDSTATR == 2`
- Obesity: `BMXBMI >= 30`
- Diabetes: `DIQ010 == 1`
- Weight used for prevalence estimates: `WTMECPRP`

## Outputs
- `outputs/table1_demographic_distribution.csv`
- `outputs/table2_risk_factors.csv`
- `outputs/table3_summary_comparison.csv`
- `outputs/analytic_dataset.csv`
- `outputs/nhanes_diabetes_obesity_tables.xlsx`

## Tables
### Table 1
Demographic distribution comparison of diabetes and obesity by:
- total adults
- age group
- sex
- race/ethnicity

### Table 2
Risk factors in clinical and person comparison:
- mean age
- mean BMI
- high waist circumference
- female percentage
- age 60+ percentage

### Table 3
Summary comparison:
- weighted prevalence
- weighted population estimate
- mean age
- mean BMI
- high waist prevalence

## Installation
This project is designed for Google Colab, but can also run in Python with:
```bash
pip install pandas requests openpyxl
```
"""

In [ ]:
# =========================
# 13. Round outputs
# =========================
def round_tables(table1, table2, table3):
    table1_out = table1.copy()
    table2_out = table2.copy()
    table3_out = table3.copy()

    # ----- Table 1 -----
    # weighted population columns as integers
    table1_out["Weighted_pop_diabetes"] = table1_out["Weighted_pop_diabetes"].round(0).astype("Int64")
    table1_out["Weighted_pop_obesity"] = table1_out["Weighted_pop_obesity"].round(0).astype("Int64")

    # counts as integers
    table1_out["Unweighted_n_diabetes"] = table1_out["Unweighted_n_diabetes"].astype("Int64")
    table1_out["Unweighted_n_obesity"] = table1_out["Unweighted_n_obesity"].astype("Int64")

    # prevalence values keep 1 decimal
    table1_out["Diabetes_prevalence_pct"] = table1_out["Diabetes_prevalence_pct"].round(1)
    table1_out["Obesity_prevalence_pct"] = table1_out["Obesity_prevalence_pct"].round(1)

    # ----- Table 2 -----
    # unweighted and weighted population as integers
    table2_out["Unweighted_n"] = table2_out["Unweighted_n"].astype("Int64")
    table2_out["Weighted_population"] = table2_out["Weighted_population"].round(0).astype("Int64")

    # remaining continuous / percentage columns rounded to 1 decimal
    cols_table2_round1 = [
        "Mean_age_years",
        "Mean_BMI",
        "High_waist_pct",
        "Female_pct",
        "Age_60plus_pct",
        "Obesity_pct_within_group",
        "Diabetes_pct_within_group",
    ]
    for col in cols_table2_round1:
        table2_out[col] = table2_out[col].round(1)

    # ----- Table 3 -----
    # make weighted population values integers only
    mask_weighted_rows = table3_out["Indicator"].isin([
        "Weighted adult population denominator",
        "Weighted population estimate with condition"
    ])
    table3_out.loc[mask_weighted_rows, "Diabetes"] = table3_out.loc[mask_weighted_rows, "Diabetes"].round(0)
    table3_out.loc[mask_weighted_rows, "Obesity"] = table3_out.loc[mask_weighted_rows, "Obesity"].round(0)

    # make unweighted sample row integers too
    mask_unweighted_row = table3_out["Indicator"] == "Adult analytic sample (unweighted)"
    table3_out.loc[mask_unweighted_row, "Diabetes"] = table3_out.loc[mask_unweighted_row, "Diabetes"].round(0)
    table3_out.loc[mask_unweighted_row, "Obesity"] = table3_out.loc[mask_unweighted_row, "Obesity"].round(0)

    # all other table 3 values rounded to 1 decimal
    mask_other_rows = ~(mask_weighted_rows | mask_unweighted_row)
    table3_out.loc[mask_other_rows, "Diabetes"] = table3_out.loc[mask_other_rows, "Diabetes"].round(1)
    table3_out.loc[mask_other_rows, "Obesity"] = table3_out.loc[mask_other_rows, "Obesity"].round(1)

    return table1_out, table2_out, table3_out

table1_out, table2_out, table3_out = round_tables(table1, table2, table3)


In [ ]:
# =========================
# Aggregated cohort table
# =========================

cohort_rows = []

def cohort_row(label, sub):
    cohort_rows.append({
        "Group": label,
        "Unweighted_n_diabetes": int(sub["diabetes"].notna().sum()),
        "Unweighted_n_obesity": int(sub["obesity"].notna().sum()),
        "Weighted_pop_diabetes": int(round(weighted_denominator(sub, "diabetes"))),
        "Weighted_pop_obesity": int(round(weighted_denominator(sub, "obesity"))),
        "Diabetes_prevalence_pct": round(weighted_prevalence(sub, "diabetes"), 1),
        "Obesity_prevalence_pct": round(weighted_prevalence(sub, "obesity"), 1),
    })

# Total adults
cohort_row("Total adults", df)

# Age groups
for g in ["18-39", "40-59", "60+"]:
    cohort_row(f"Age {g}", df[df["age_group"] == g])

# Sex
for g in ["Male", "Female"]:
    cohort_row(g, df[df["sex"] == g])

# Race
for g in [
    "Mexican American",
    "Other Hispanic",
    "Non-Hispanic White",
    "Non-Hispanic Black",
]:
    cohort_row(g, df[df["race_eth"] == g])

cohort_table = pd.DataFrame(cohort_rows)
print(cohort_table)


In [ ]:
# =========================
# Save tables to local folder
# =========================

table1_csv = os.path.join(OUTPUT_DIR, "table1_demographic_distribution.csv")
table2_csv = os.path.join(OUTPUT_DIR, "table2_risk_factors.csv")
table3_csv = os.path.join(OUTPUT_DIR, "table3_summary_comparison.csv")
analytic_csv = os.path.join(OUTPUT_DIR, "analytic_dataset.csv")

table1_out.to_csv(table1_csv, index=False)
table2_out.to_csv(table2_csv, index=False)
table3_out.to_csv(table3_csv, index=False)
df.to_csv(analytic_csv, index=False)

print("Files saved to:")
print(table1_csv)
print(table2_csv)
print(table3_csv)
print(analytic_csv)

In [ ]:
import pandas as pd
import os

# =========================
# Output directory
# =========================
OUTPUT_DIR = "/Users/haoqimacbookpro/Desktop/Ling/2026 Spring"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================
# Aggregated cohort table
# =========================
cohort_table = pd.DataFrame({
    "Row":all,
    "Group":[
        "Total adults",
        "Age 18–39",
        "Age 40–59",
        "Age 60+",
        "Male",
        "Female",
        "Mexican American",
        "Other Hispanic",
        "Non-Hispanic White",
        "Non-Hispanic Black"
    ],
    "Unweighted_n_diabetes":["","","","","","","","","",""],
    "Unweighted_n_obesity":["","","","","","","","","",""],
    "Weighted_pop_diabetes":["","","","","","","","","",""],
    "Weighted_pop_obesity":["","","","","","","","","",""],
    "Diabetes_prevalence_pct":["","","","","","","","","",""],
    "Obesity_prevalence_pct":["","","","","","","","","",""],
})

# =========================
# Save CSV (full cohort table)
# =========================
csv_path = os.path.join(OUTPUT_DIR,"aggregated_cohort_table.csv")
cohort_table.to_csv(csv_path,index=False)

print("CSV saved to:")
print(csv_path)

# =========================
# Create Markdown version (for GitHub)
# =========================
markdown_path = os.path.join(OUTPUT_DIR,"aggregated_cohort_table.md")

with open(markdown_path,"w") as f:
    f.write("# Aggregated Cohort Table (NHANES Diabetes vs Obesity)\n\n")
    f.write(cohort_table.to_markdown(index=False))

print("Markdown table saved to:")
print(markdown_path)

# =========================
# Create Word-ready table
# =========================
!pip install python-docx
from docx import Document

doc = Document()
doc.add_heading("Aggregated Cohort Table", level=1)

table = doc.add_table(rows=len(cohort_table)+1, cols=len(cohort_table.columns))

# header
for j,col in enumerate(cohort_table.columns):
    table.rows[0].cells[j].text = col

# rows
for i,row in cohort_table.iterrows():
    for j,val in enumerate(row):
        table.rows[i+1].cells[j].text = str(val)

doc_path = os.path.join(OUTPUT_DIR,"aggregated_cohort_table.docx")
doc.save(doc_path)

print("Word table saved to:")
print(doc_path)


In [ ]:
import pandas as pd
import os

# =========================
# Output directory
# =========================
OUTPUT_DIR = "/Users/haoqimacbookpro/Desktop/Ling/2026 Spring"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================
# Aggregated cohort table
# =========================
cohort_table = pd.DataFrame({
    "Row":all,
    "Group":[
        "Total adults",
        "Age 18–39",
        "Age 40–59",
        "Age 60+",
        "Male",
        "Female",
        "Mexican American",
        "Other Hispanic",
        "Non-Hispanic White",
        "Non-Hispanic Black"
    ],
    "Unweighted_n_diabetes":["","","","","","","","","",""],
    "Unweighted_n_obesity":["","","","","","","","","",""],
    "Weighted_pop_diabetes":["","","","","","","","","",""],
    "Weighted_pop_obesity":["","","","","","","","","",""],
    "Diabetes_prevalence_pct":["","","","","","","","","",""],
    "Obesity_prevalence_pct":["","","","","","","","","",""]
})

# =========================
# Save CSV (full cohort table)
# =========================
csv_path = os.path.join(OUTPUT_DIR,"aggregated_cohort_table.csv")
cohort_table.to_csv(csv_path,index=False)

print("CSV saved to:")
print(csv_path)

# =========================
# Create Markdown version (for GitHub)
# =========================
markdown_path = os.path.join(OUTPUT_DIR,"aggregated_cohort_table.md")

with open(markdown_path,"w") as f:
    f.write("# Aggregated Cohort Table (NHANES Diabetes vs Obesity)\n\n")
    f.write(cohort_table.to_markdown(index=False))

print("Markdown table saved to:")
print(markdown_path)

# =========================
# Create Word-ready table
# =========================
from docx import Document

doc = Document()
doc.add_heading("Aggregated Cohort Table", level=1)

table = doc.add_table(rows=len(cohort_table)+1, cols=len(cohort_table.columns))

# header
for j,col in enumerate(cohort_table.columns):
    table.rows[0].cells[j].text = col

# rows
for i,row in cohort_table.iterrows():
    for j,val in enumerate(row):
        table.rows[i+1].cells[j].text = str(val)

doc_path = os.path.join(OUTPUT_DIR,"aggregated_cohort_table.docx")
doc.save(doc_path)

print("Word table saved to:")
print(doc_path)

In [ ]:
import os
import pandas as pd
import numpy as np

# =========================================================
# Local save path on your Mac
# =========================================================
OUTPUT_DIR = "/Users/haoqimacbookpro/Desktop/Ling/2026 Spring"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================================================
# Assumes df already exists from your NHANES script
# and includes these columns:
# RIDAGEYR, RIAGENDR, RIDRETH3, RIDSTATR, WTMECPRP,
# BMXBMI, DIQ010, obesity, diabetes, age_group, sex, race_eth
# =========================================================

WEIGHT = "WTMECPRP"

def weighted_prevalence(data: pd.DataFrame, outcome: str, weight: str = WEIGHT) -> float:
    temp = data[[outcome, weight]].dropna()
    if temp.empty:
        return np.nan
    return 100 * (temp[outcome] * temp[weight]).sum() / temp[weight].sum()

def weighted_denominator(data: pd.DataFrame, outcome: str, weight: str = WEIGHT) -> float:
    temp = data[[outcome, weight]].dropna()
    if temp.empty:
        return np.nan
    return temp[weight].sum()

def build_aggregated_cohort_table(df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    def add_row(label: str, sub: pd.DataFrame):
        rows.append({
            "Group": label,
            "Unweighted_n_diabetes": int(sub["diabetes"].notna().sum()),
            "Unweighted_n_obesity": int(sub["obesity"].notna().sum()),
            "Weighted_pop_diabetes": weighted_denominator(sub, "diabetes"),
            "Weighted_pop_obesity": weighted_denominator(sub, "obesity"),
            "Diabetes_prevalence_pct": weighted_prevalence(sub, "diabetes"),
            "Obesity_prevalence_pct": weighted_prevalence(sub, "obesity"),
        })

    # Total
    add_row("Total adults", df)

    # Age groups
    for g in ["18-39", "40-59", "60+"]:
        add_row(f"Age {g}", df[df["age_group"] == g])

    # Sex
    for g in ["Male", "Female"]:
        add_row(g, df[df["sex"] == g])

    # Race/ethnicity
    for g in [
        "Mexican American",
        "Other Hispanic",
        "Non-Hispanic White",
        "Non-Hispanic Black",
        "Non-Hispanic Asian",
        "Other / Multiracial"
    ]:
        add_row(g, df[df["race_eth"] == g])

    out = pd.DataFrame(rows)

    # Round exactly the way you asked
    out["Weighted_pop_diabetes"] = out["Weighted_pop_diabetes"].round(0).astype("Int64")
    out["Weighted_pop_obesity"] = out["Weighted_pop_obesity"].round(0).astype("Int64")
    out["Diabetes_prevalence_pct"] = out["Diabetes_prevalence_pct"].round(1)
    out["Obesity_prevalence_pct"] = out["Obesity_prevalence_pct"].round(1)

    return out

# =========================================================
# Build the real cohort table
# =========================================================
aggregated_cohort_table = build_aggregated_cohort_table(df)

# Add row number column
aggregated_cohort_table.insert(0, "Row", range(1, len(aggregated_cohort_table) + 1))

# First 10 rows
aggregated_cohort_table_head10 = aggregated_cohort_table.head(10).copy()

# =========================================================
# Save files
# =========================================================
csv_full_path = os.path.join(OUTPUT_DIR, "aggregated_cohort_table.csv")
csv_head10_path = os.path.join(OUTPUT_DIR, "aggregated_cohort_table_head10.csv")
md_path = os.path.join(OUTPUT_DIR, "aggregated_cohort_table.md")

aggregated_cohort_table.to_csv(csv_full_path, index=False)
aggregated_cohort_table_head10.to_csv(csv_head10_path, index=False)

with open(md_path, "w", encoding="utf-8") as f:
    f.write("# Aggregated Cohort Table\n\n")
    f.write(aggregated_cohort_table_head10.to_markdown(index=False))
    f.write("\n\n")
    f.write(
        "*Footnote.* Data source: NHANES 2017–March 2020 pre-pandemic cohort "
        "merged from `P_DEMO`, `P_BMX`, and `P_DIQ` by `SEQN`. Adults aged 18 years "
        "and older with MEC examination status (`RIDSTATR == 2`) were included. "
        "Diabetes was defined as `DIQ010 == 1`. Obesity was defined as `BMXBMI >= 30`. "
        "Weighted population estimates were calculated using `WTMECPRP` and rounded "
        "to whole numbers. Prevalence percentages were calculated as weighted proportions "
        "within each subgroup."
    )

print("Saved files:")
print(csv_full_path)
print(csv_head10_path)
print(md_path)

print("\nPreview:")
print(aggregated_cohort_table_head10)

In [ ]:
table3_out.to_csv(_csv, index=False)

In [ ]:
import os
import pandas as pd
import numpy as np

# =========================================================
# Local save path on your Mac
# =========================================================
OUTPUT_DIR = "/Users/haoqimacbookpro/Desktop/Ling/2026 Spring"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================================================
# Assumes df already exists from your NHANES script
# and includes these columns:
# RIDAGEYR, RIAGENDR, RIDRETH3, RIDSTATR, WTMECPRP,
# BMXBMI, DIQ010, obesity, diabetes, age_group, sex, race_eth
# =========================================================

WEIGHT = "WTMECPRP"

def weighted_prevalence(data: pd.DataFrame, outcome: str, weight: str = WEIGHT) -> float:
    temp = data[[outcome, weight]].dropna()
    if temp.empty:
        return np.nan
    return 100 * (temp[outcome] * temp[weight]).sum() / temp[weight].sum()

def weighted_denominator(data: pd.DataFrame, outcome: str, weight: str = WEIGHT) -> float:
    temp = data[[outcome, weight]].dropna()
    if temp.empty:
        return np.nan
    return temp[weight].sum()

def build_aggregated_cohort_table(df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    def add_row(label: str, sub: pd.DataFrame):
        rows.append({
            "Group": label,
            "Unweighted_n_diabetes": int(sub["diabetes"].notna().sum()),
            "Unweighted_n_obesity": int(sub["obesity"].notna().sum()),
            "Weighted_pop_diabetes": weighted_denominator(sub, "diabetes"),
            "Weighted_pop_obesity": weighted_denominator(sub, "obesity"),
            "Diabetes_prevalence_pct": weighted_prevalence(sub, "diabetes"),
            "Obesity_prevalence_pct": weighted_prevalence(sub, "obesity"),
        })

    # Total
    add_row("Total adults", df)

    # Age groups
    for g in ["18-39", "40-59", "60+"]:
        add_row(f"Age {g}", df[df["age_group"] == g])

    # Sex
    for g in ["Male", "Female"]:
        add_row(g, df[df["sex"] == g])

    # Race/ethnicity
    for g in [
        "Mexican American",
        "Other Hispanic",
        "Non-Hispanic White",
        "Non-Hispanic Black",
        "Non-Hispanic Asian",
        "Other / Multiracial"
    ]:
        add_row(g, df[df["race_eth"] == g])

    out = pd.DataFrame(rows)

    # Round exactly the way you asked
    out["Weighted_pop_diabetes"] = out["Weighted_pop_diabetes"].round(0).astype("Int64")
    out["Weighted_pop_obesity"] = out["Weighted_pop_obesity"].round(0).astype("Int64")
    out["Diabetes_prevalence_pct"] = out["Diabetes_prevalence_pct"].round(1)
    out["Obesity_prevalence_pct"] = out["Obesity_prevalence_pct"].round(1)

    return out

# =========================================================
# Build the real cohort table
# =========================================================
aggregated_cohort_table = build_aggregated_cohort_table(df)

# Add row number column
aggregated_cohort_table.insert(0, "Row", range(1, len(aggregated_cohort_table) + 1))

# First 10 rows
aggregated_cohort_table_head10 = aggregated_cohort_table.head(10).copy()

# =========================================================
# Save files
# =========================================================
csv_full_path = os.path.join(OUTPUT_DIR, "aggregated_cohort_table.csv")
csv_head10_path = os.path.join(OUTPUT_DIR, "aggregated_cohort_table_head10.csv")
md_path = os.path.join(OUTPUT_DIR, "aggregated_cohort_table.md")

aggregated_cohort_table.to_csv(csv_full_path, index=False)
aggregated_cohort_table_head10.to_csv(csv_head10_path, index=False)

with open(md_path, "w", encoding="utf-8") as f:
    f.write("# Aggregated Cohort Table\n\n")
    f.write(aggregated_cohort_table_head10.to_markdown(index=False))
    f.write("\n\n")
    f.write(
        "*Footnote.* Data source: NHANES 2017–March 2020 pre-pandemic cohort "
        "merged from `P_DEMO`, `P_BMX`, and `P_DIQ` by `SEQN`. Adults aged 18 years "
        "and older with MEC examination status (`RIDSTATR == 2`) were included. "
        "Diabetes was defined as `DIQ010 == 1`. Obesity was defined as `BMXBMI >= 30`. "
        "Weighted population estimates were calculated using `WTMECPRP` and rounded "
        "to whole numbers. Prevalence percentages were calculated as weighted proportions "
        "within each subgroup."
    )

print("Saved files:")
print(csv_full_path)
print(csv_head10_path)
print(md_path)

print("\nPreview:")
print(aggregated_cohort_table_head10)

In [ ]:
save

In [ ]:
from google.colab import files
import os

# Define the output path for the full aggregated cohort table
output_filepath_full = os.path.join(OUTPUT_DIR, "aggregated_cohort_table_full.csv")

# Save the full aggregated cohort table to a CSV file
aggregated_cohort_table.to_csv(output_filepath_full, index=False)
print(f"Full aggregated cohort table saved to: {output_filepath_full}")

# Trigger the download for the full table
files.download(output_filepath_full)


In [ ]:
from google.colab import files

output_filepath = os.path.join(OUTPUT_DIR, "aggregated_cohort_table.csv")
files.download(output_filepath)

In [ ]:
output_filepath = os.path.join(OUTPUT_DIR, "aggregated_cohort_table.csv")
aggregated_cohort_table.to_csv(output_filepath, index=False)
print(f"Aggregated cohort table saved to: {output_filepath}")

# Task
Analyze the relationships between demographic and clinical factors and the prevalence of diabetes and obesity in the NHANES dataset. This involves: selecting relevant features (age, sex, BMI, waist circumference, race/ethnicity) and target variables (diabetes, obesity) from the main DataFrame; handling missing values; encoding categorical variables; scaling numerical features; training two classification models, one for predicting diabetes and another for predicting obesity; analyzing feature importance from these models to identify influential factors; visualizing feature importance for both conditions using bar charts with appropriate legends; and finally, summarizing the insights gained regarding the main comparisons and relationships between diabetes and obesity based on the feature importance analysis.

## Prepare Data for Machine Learning

### Subtask:
Select relevant features (e.g., age, sex, BMI, waist circumference, race/ethnicity) and target variables (diabetes, obesity) from the main DataFrame. Handle missing values, encode categorical variables, and scale numerical features as necessary to prepare the data for machine learning models.


**Reasoning**:
The subtask is to prepare data for machine learning by selecting relevant features and target variables, handling missing values, encoding categorical variables, and scaling numerical features. The first step is to define the feature and target columns and create a new DataFrame with only these columns.



In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# 1. Define a list of feature columns and target columns
feature_columns = ['RIDAGEYR', 'sex', 'race_eth', 'age_group', 'BMXBMI', 'BMXWAIST']
target_columns = ['diabetes', 'obesity']

# 2. Create a new DataFrame containing only these selected feature and target columns
ml_df = df[feature_columns + target_columns].copy()

print(f"Initial ML DataFrame shape: {ml_df.shape}")

**Reasoning**:
The next step is to handle missing values by dropping any rows from the `ml_df` DataFrame that contain missing values in any of the selected feature or target columns to ensure complete data for machine learning models.



In [ ]:
ml_df.dropna(inplace=True)
print(f"ML DataFrame shape after dropping NaNs: {ml_df.shape}")

**Reasoning**:
The next step is to separate the `ml_df` into features (X) and target variables (y_diabetes, y_obesity) as per the instructions, which is crucial for machine learning model training.



In [ ]:
X = ml_df[feature_columns]
y_diabetes = ml_df['diabetes']
y_obesity = ml_df['obesity']

print(f"Shape of features (X): {X.shape}")
print(f"Shape of diabetes target (y_diabetes): {y_diabetes.shape}")
print(f"Shape of obesity target (y_obesity): {y_obesity.shape}")

**Reasoning**:
According to the instructions, the next step is to explicitly identify the categorical and numerical feature columns within the `X` DataFrame. This distinction is crucial for applying appropriate preprocessing steps like one-hot encoding and standard scaling.



In [ ]:
categorical_features = ['sex', 'race_eth', 'age_group']
numerical_features = ['RIDAGEYR', 'BMXBMI', 'BMXWAIST']

print(f"Categorical features: {categorical_features}")
print(f"Numerical features: {numerical_features}")

**Reasoning**:
The next instruction is to apply one-hot encoding to the categorical feature columns. This will convert categorical variables into a format suitable for machine learning algorithms.



In [ ]:
X = pd.get_dummies(X, columns=categorical_features, drop_first=True)
print(f"Shape of features (X) after one-hot encoding: {X.shape}")

**Reasoning**:
The final step in data preparation, as per the instructions, is to apply standard scaling to the numerical feature columns. This will normalize the numerical data, which is beneficial for many machine learning algorithms.



In [ ]:
scaler = StandardScaler()
X[numerical_features] = scaler.fit_transform(X[numerical_features])

print(f"Shape of features (X) after scaling numerical features: {X.shape}")

## Train Classification Models

### Subtask:
Train two separate classification models, one for predicting diabetes and another for predicting obesity, using the prepared features.


**Reasoning**:
The subtask requires training two classification models. The first step is to import the necessary `LogisticRegression` class from `sklearn.linear_model`.



In [ ]:
from sklearn.linear_model import LogisticRegression

**Reasoning**:
Now that `LogisticRegression` is imported, the next step is to instantiate the two models, `diabetes_model` and `obesity_model`, and then train each model using the prepared features `X` and their respective target variables `y_diabetes` and `y_obesity`.



In [ ]:
diabetes_model = LogisticRegression(max_iter=1000, random_state=42)
obesity_model = LogisticRegression(max_iter=1000, random_state=42)

diabetes_model.fit(X, y_diabetes)
obesity_model.fit(X, y_obesity)

print("Diabetes model trained.")
print("Obesity model trained.")

## Analyze Feature Importance

### Subtask:
Extract feature importance scores (coefficients) from the trained logistic regression models for both diabetes and obesity.


**Reasoning**:
The subtask requires extracting feature importance scores (coefficients) from the trained logistic regression models. I will access the `coef_` attribute of both `diabetes_model` and `obesity_model`, convert them into Pandas Series with feature names as indices, and then print them.



In [ ]:
diabetes_feature_importance = pd.Series(diabetes_model.coef_[0], index=X.columns)
obesity_feature_importance = pd.Series(obesity_model.coef_[0], index=X.columns)

print("\nDiabetes Feature Importance (Coefficients):")
print(diabetes_feature_importance)

print("\nObesity Feature Importance (Coefficients):")
print(obesity_feature_importance)

**Reasoning**:
The next logical step in analyzing feature importance is to visualize the extracted coefficients to gain better insights into which features are most influential for each model. This will involve creating bar charts for both diabetes and obesity feature importances.



In [ ]:
import matplotlib.pyplot as plt

# Sort feature importance for better visualization
diabetes_feature_importance = diabetes_feature_importance.sort_values(ascending=True)
obesity_feature_importance = obesity_feature_importance.sort_values(ascending=True)

# Create subplots
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Plot Diabetes Feature Importance
diabetes_feature_importance.plot(kind='barh', ax=axes[0], color='skyblue')
axes[0].set_title('Feature Importance for Diabetes Prediction')
axes[0].set_xlabel('Coefficient Value')
axes[0].set_ylabel('Features')
axes[0].grid(axis='x', linestyle='--')

# Plot Obesity Feature Importance
obesity_feature_importance.plot(kind='barh', ax=axes[1], color='lightcoral')
axes[1].set_title('Feature Importance for Obesity Prediction')
axes[1].set_xlabel('Coefficient Value')
axes[1].set_ylabel('Features')
axes[1].grid(axis='x', linestyle='--')

plt.tight_layout()
plt.show()


### Summary of Feature Importance Analysis

**Relationships between Diabetes and Obesity:**

From the feature importance plots, a clear distinction and overlap in the drivers of diabetes and obesity can be observed:

- **BMXBMI (BMI) is the strongest predictor for Obesity:** The coefficient for `BMXBMI` in the obesity model is overwhelmingly high, as expected, given that obesity is defined by BMI. This indicates a very direct and strong relationship.
- **BMXWAIST (Waist Circumference) and RIDAGEYR (Age) are strong predictors for Diabetes:** Waist circumference and age are highly influential factors in predicting diabetes, with positive coefficients. This suggests that as waist circumference and age increase, the likelihood of diabetes also increases significantly.
- **BMI's role in Diabetes:** While `BMXBMI` has a negative coefficient for diabetes, this might be counterintuitive at first glance. However, since `BMXWAIST` is a very strong positive predictor for diabetes, and `BMXBMI` is still included, it implies that among individuals with a certain waist circumference, BMI itself might not be as directly differentiating for diabetes, or there might be some interaction effects not fully captured by this linear model. It's more likely that multicollinearity between BMI and Waist Circumference means that once waist circumference is accounted for, the marginal effect of BMI alone shifts.

**Influential Factors for Diabetes Prediction:**

- **Positive Predictors:**
  - `age_group_60+` and `age_group_40-59`: Older age groups show a strong positive association with diabetes, indicating that age is a major risk factor.
  - `BMXWAIST`: Higher waist circumference is a significant predictor of diabetes.
  - `RIDAGEYR`: Increasing age (continuous) is also a positive predictor.
- **Negative Predictors:**
  - `race_eth_Non-Hispanic White`, `race_eth_Non-Hispanic Black`, `race_eth_Other Hispanic`, `race_eth_Other / Multiracial`: These racial/ethnic groups show negative coefficients relative to the baseline (Mexican American, which was dropped by `drop_first=True` in `pd.get_dummies` for race/ethnicity), suggesting lower odds of diabetes compared to the reference group.

**Influential Factors for Obesity Prediction:**

- **Positive Predictors:**
  - `BMXBMI`: As expected, BMI is by far the most dominant positive predictor of obesity.
  - `BMXWAIST`: Waist circumference also shows a positive association, though much smaller than BMI.
  - `race_eth_Non-Hispanic Black`: This group shows a very slight positive association with obesity compared to the baseline.
- **Negative Predictors:**
  - `race_eth_Non-Hispanic Asian`: This group shows a notable negative association with obesity compared to the baseline.
  - `sex_Male`: Being male shows a negative association with obesity, implying females have higher odds of obesity compared to males.
  - `RIDAGEYR`, `age_group_40-59`, `age_group_60+`: Interestingly, age and older age groups show slight negative coefficients, suggesting that after controlling for BMI and waist circumference, older individuals might have slightly lower odds of being classified as obese in this model, or it might be related to other demographic distributions. The definition of obesity (`BMXBMI >= 30`) itself is age-independent.

**Overall:** Both diabetes and obesity are complex conditions influenced by a mix of demographic and anthropometric factors. While BMI is the primary determinant for obesity, diabetes risk is more strongly linked to age and waist circumference. There are also important racial/ethnic and sex differences in the prevalence and risk factors for both conditions.

### Summary of Feature Importance Analysis

**Relationships between Diabetes and Obesity:**

From the feature importance plots, a clear distinction and overlap in the drivers of diabetes and obesity can be observed:

- **BMXBMI (BMI) is the strongest predictor for Obesity:** The coefficient for `BMXBMI` in the obesity model is overwhelmingly high, as expected, given that obesity is defined by BMI. This indicates a very direct and strong relationship.
- **BMXWAIST (Waist Circumference) and RIDAGEYR (Age) are strong predictors for Diabetes:** Waist circumference and age are highly influential factors in predicting diabetes, with positive coefficients. This suggests that as waist circumference and age increase, the likelihood of diabetes also increases significantly.
- **BMI's role in Diabetes:** While `BMXBMI` has a negative coefficient for diabetes, this might be counterintuitive at first glance. However, since `BMXWAIST` is a very strong positive predictor for diabetes, and `BMXBMI` is still included, it implies that among individuals with a certain waist circumference, BMI itself might not be as directly differentiating for diabetes, or there might be some interaction effects not fully captured by this linear model. It's more likely that multicollinearity between BMI and Waist Circumference means that once waist circumference is accounted for, the marginal effect of BMI alone shifts.

**Influential Factors for Diabetes Prediction:**

- **Positive Predictors:**
  - `age_group_60+` and `age_group_40-59`: Older age groups show a strong positive association with diabetes, indicating that age is a major risk factor.
  - `BMXWAIST`: Higher waist circumference is a significant predictor of diabetes.
  - `RIDAGEYR`: Increasing age (continuous) is also a positive predictor.
- **Negative Predictors:**
  - `race_eth_Non-Hispanic White`, `race_eth_Non-Hispanic Black`, `race_eth_Other Hispanic`, `race_eth_Other / Multiracial`: These racial/ethnic groups show negative coefficients relative to the baseline (Mexican American, which was dropped by `drop_first=True` in `pd.get_dummies` for race/ethnicity), suggesting lower odds of diabetes compared to the reference group.

**Influential Factors for Obesity Prediction:**

- **Positive Predictors:**
  - `BMXBMI`: As expected, BMI is by far the most dominant positive predictor of obesity.
  - `BMXWAIST`: Waist circumference also shows a positive association, though much smaller than BMI.
  - `race_eth_Non-Hispanic Black`: This group shows a very slight positive association with obesity compared to the baseline.
- **Negative Predictors:**
  - `race_eth_Non-Hispanic Asian`: This group shows a notable negative association with obesity compared to the baseline.
  - `sex_Male`: Being male shows a negative association with obesity, implying females have higher odds of obesity compared to males.
  - `RIDAGEYR`, `age_group_40-59`, `age_group_60+`: Interestingly, age and older age groups show slight negative coefficients, suggesting that after controlling for BMI and waist circumference, older individuals might have slightly lower odds of being classified as obese in this model, or it might be related to other demographic distributions. The definition of obesity (`BMXBMI >= 30`) itself is age-independent.

**Overall:** Both diabetes and obesity are complex conditions influenced by a mix of demographic and anthropometric factors. While BMI is the primary determinant for obesity, diabetes risk is more strongly linked to age and waist circumference. There are also important racial/ethnic and sex differences in the prevalence and risk factors for both conditions.

## Final Task

### Subtask:
Summarize the insights gained from the machine learning analysis regarding the main comparisons and relationships between diabetes and obesity, based on feature importance and model findings.


## Summary:

### Data Analysis Key Findings

*   **Data Preparation:** The initial dataset of 8965 observations was refined to 8428 observations after dropping rows with missing values. Relevant features such as age, sex, BMI, waist circumference, and race/ethnicity, along with target variables (diabetes, obesity), were selected. Categorical features were one-hot encoded, expanding the feature set from 6 to 11 columns, and numerical features were standardized.
*   **Model Training:** Two separate Logistic Regression models were successfully trained: one for predicting diabetes and another for predicting obesity, using the preprocessed features.
*   **Feature Importance for Diabetes Prediction:**
    *   **Strong Positive Predictors:** `BMXWAIST` (waist circumference) and `RIDAGEYR` (age) were identified as highly influential positive predictors, with older age groups (`age_group_60+`, `age_group_40-59`) also showing strong positive associations.
    *   **Counter-intuitive BMI Role:** `BMXBMI` (BMI) displayed a negative coefficient for diabetes, which was noted as potentially due to multicollinearity with `BMXWAIST`, suggesting that once waist circumference is accounted for, the marginal effect of BMI alone shifts.
    *   **Racial/Ethnic Differences:** Several racial/ethnic groups (`Non-Hispanic White`, `Non-Hispanic Black`, `Other Hispanic`, `Other / Multiracial`) showed negative coefficients relative to the Mexican American baseline, implying lower odds of diabetes for these groups.
*   **Feature Importance for Obesity Prediction:**
    *   **Dominant Predictor:** `BMXBMI` (BMI) was overwhelmingly the strongest positive predictor for obesity, as expected, with a coefficient of approximately 17.12.
    *   **Secondary Predictor:** `BMXWAIST` (waist circumference) also showed a positive association, though significantly smaller than BMI, with a coefficient of approximately 1.09.
    *   **Sex and Racial/Ethnic Differences:** Being male (`sex_Male`) was negatively associated with obesity, implying higher odds for females. `race_eth_Non-Hispanic Asian` showed a notable negative association compared to the baseline, while `race_eth_Non-Hispanic Black` showed a slight positive association.
    *   **Age's Complex Role:** Age-related features (`RIDAGEYR`, `age_group_40-59`, `age_group_60+`) exhibited slight negative coefficients, suggesting that after controlling for BMI and waist circumference, older individuals might have marginally lower odds of being classified as obese within this model.

### Insights or Next Steps

*   **Distinct Primary Drivers:** While obesity is primarily driven by BMI, diabetes risk is more strongly linked to age and waist circumference. This highlights the importance of targeted interventions: addressing BMI for general obesity prevention, and focusing on age-related screenings and managing central adiposity (waist circumference) for diabetes prevention.
*   **Investigate Multicollinearity and Interactions:** The observed negative coefficient for BMI in diabetes prediction, despite its known association, suggests potential multicollinearity with waist circumference. Future analysis should explore interaction effects between BMI and waist circumference, or utilize models less sensitive to multicollinearity, to better understand their individual and combined impacts on diabetes risk.
